# 04 — Regresores externos: tipos BCE + IBEX 35

Añadimos al Prophet del NB 02 dos regresores macroeconómicos exógenos:

1. **Tipo de interés BCE** (refinanciación principal) — captura coste de financiación bancario. Frecuencia: cambios puntuales del Consejo de Gobierno.
2. **IBEX 35** — captura el estado general del mercado español. Frecuencia: diaria.

Objetivo: cuantificar si la información macro mejora el MAE/RMSE respecto al Prophet+OPA del NB 02 (referencia: MAE 0.583, RMSE 0.635, MAPE 18.65%).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from prophet import Prophet

from src.data_loader import get_sab, get_ibex, OPA_BBVA_EVENTS

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Cargar SAB e IBEX

In [ ]:
sab  = get_sab(period='5y')
ibex = get_ibex(period='5y')
for d in (sab, ibex):
    d['Date'] = pd.to_datetime(d['Date'])
print('SAB:', len(sab), '| IBEX:', len(ibex))

## 2. Tipos BCE (refinanciación principal)

Cargamos manualmente las decisiones del Consejo de Gobierno del BCE (fuente: ECB Data Portal — *Main refinancing operations rate*). En producción se descargaría vía API o `pandas-datareader`.

In [ ]:
# Tipo de refinanciación principal del BCE -- fechas efectivas de cambio
bce_changes = pd.DataFrame({
    'Date': pd.to_datetime([
        '2021-05-18', '2022-07-27', '2022-09-14', '2022-11-02', '2022-12-21',
        '2023-02-08', '2023-03-22', '2023-05-10', '2023-06-21', '2023-08-02',
        '2023-09-20', '2024-06-12', '2024-09-18', '2024-10-23', '2024-12-18',
        '2025-01-29', '2025-03-12', '2025-04-23', '2025-06-11', '2025-09-17',
    ]),
    'tipo_bce': [
        0.00, 0.50, 1.25, 2.00, 2.50,
        3.00, 3.50, 3.75, 4.00, 4.25,
        4.50, 4.25, 3.65, 3.40, 3.15,
        2.90, 2.65, 2.40, 2.15, 2.00,
    ]
})
bce_changes.tail()

In [ ]:
# expandir a frecuencia diaria con forward fill
date_range = pd.date_range(start=sab['Date'].min(), end=sab['Date'].max() + pd.Timedelta(days=120), freq='D')
bce_daily = pd.DataFrame({'Date': date_range}).merge(bce_changes, on='Date', how='left').ffill()
bce_daily.tail()

## 3. Construir dataset Prophet con regresores

In [ ]:
df = sab[['Date','Close']].rename(columns={'Date':'ds','Close':'y'}).copy()
df = df.merge(ibex[['Date','Close']].rename(columns={'Date':'ds','Close':'ibex'}), on='ds', how='left')
df = df.merge(bce_daily.rename(columns={'Date':'ds'}), on='ds', how='left')
df = df.dropna().reset_index(drop=True)
print('Filas tras merge:', len(df))
df.tail()

In [ ]:
HORIZON = 90
train = df.iloc[:-HORIZON].copy()
test  = df.iloc[-HORIZON:].copy()
print(f'Train: {len(train)} | Test: {len(test)}')

## 4. MLflow

In [ ]:
mlflow.set_tracking_uri('file:///' + (ROOT / 'mlruns').as_posix())
mlflow.set_experiment('sab_forecast_regresores')

## 5. Prophet con OPA + IBEX + BCE

In [ ]:
with mlflow.start_run(run_name='prophet_opa_ibex_bce'):
    m = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True,
        holidays=OPA_BBVA_EVENTS,
        interval_width=0.80,
        changepoint_prior_scale=0.10,
    )
    m.add_country_holidays(country_name='ES')
    m.add_regressor('ibex',     prior_scale=0.5,  standardize=True)
    m.add_regressor('tipo_bce', prior_scale=0.5,  standardize=True)
    m.fit(train)

    # para predecir el test necesitamos ibex y tipo_bce en esas fechas — los tenemos en `test`
    future = test[['ds','ibex','tipo_bce']].copy()
    fcst = m.predict(future)

    eval_df = test.merge(fcst[['ds','yhat','yhat_lower','yhat_upper']], on='ds')
    mae  = (eval_df['y'] - eval_df['yhat']).abs().mean()
    rmse = np.sqrt(((eval_df['y'] - eval_df['yhat'])**2).mean())
    mape = ((eval_df['y'] - eval_df['yhat']).abs() / eval_df['y']).mean() * 100

    mlflow.log_param('model', 'prophet_opa_ibex_bce')
    mlflow.log_param('horizon', HORIZON)
    mlflow.log_param('regressors', 'ibex,tipo_bce')
    mlflow.log_metric('mae',  mae)
    mlflow.log_metric('rmse', rmse)
    mlflow.log_metric('mape', mape)

    print(f'OPA+IBEX+BCE  ·  MAE: {mae:.3f}  RMSE: {rmse:.3f}  MAPE: {mape:.2f}%')

## 6. Comparativa con NB 02

In [ ]:
# Referencia NB 02 (Prophet + OPA holidays, sin regresores)
ref = {'mae': 0.583, 'rmse': 0.635, 'mape': 18.65}

resumen = pd.DataFrame({
    'Modelo': ['Prophet baseline (NB02)', 'Prophet + OPA holidays (NB02)', 'Prophet + OPA + IBEX + BCE (NB04)'],
    'MAE':    [0.644, ref['mae'], mae],
    'RMSE':   [0.691, ref['rmse'], rmse],
    'MAPE %': [20.56, ref['mape'], mape],
}).set_index('Modelo')
resumen

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(train['ds'], train['y'], color='black', label='Train', linewidth=1)
ax.plot(test['ds'],  test['y'],  color='black', linestyle='--', label='Real (test)', linewidth=1.5)
ax.plot(eval_df['ds'], eval_df['yhat'], color='tab:purple', label=f'+ OPA+IBEX+BCE (RMSE {rmse:.2f})')
ax.fill_between(eval_df['ds'], eval_df['yhat_lower'], eval_df['yhat_upper'], color='tab:purple', alpha=0.15)
ax.set_title('SAB.MC — Forecast 90d con regresores macro')
ax.set_xlabel('Fecha'); ax.set_ylabel('EUR')
ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

In [ ]:
# Guardar previsión enriquecida
out = ROOT / 'data' / 'sab_forecast_regresores_90d.csv'
eval_df[['ds','yhat','yhat_lower','yhat_upper']].to_csv(out, index=False)
print(f'Forecast guardado: {out}')